In [8]:
# ─── 1. IMPORTS ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from google.colab import drive

In [9]:
# ─── 3. LEITURA E IDENTIFICAÇÃO DAS PLANILHAS ─────────────────────────────────
# ⚙️  Ajuste PASTA_DRIVE para o caminho da pasta no seu Drive onde estão os CSVs.
# Exemplo: 'Meu Drive/dados_acidentes'  ou  'Meu Drive'  se estiverem na raiz.
PASTA_DRIVE = 'Meu Drive'   # ← altere aqui se necessário

ARQUIVOS_ESPERADOS = [
    '2023_famar.csv',
    '2024_famar.csv',
    '2023_fumes.csv',
    '2024_fumes.csv',
]

def identificar_instituicao(nome_arquivo: str) -> str:
    nome = nome_arquivo.lower()
    if 'fumes' in nome:
        return 'FUMES'
    elif 'famar' in nome:
        return 'FAMAR'
    raise ValueError(f'Não foi possível identificar a instituição em: {nome_arquivo}')

planilhas = {}   # {'nome_arquivo': DataFrame}

for nome in ARQUIVOS_ESPERADOS:
    caminho = f'{nome}'
    try:
        df = pd.read_csv(caminho)
    except FileNotFoundError:
        print(f'[AVISO] Arquivo não encontrado: {caminho}')
        continue

    df['instituicao'] = identificar_instituicao(nome)
    df['arquivo_origem'] = nome

    # Converter coluna de data (formato yyyy-dd-MM hh:mm:ss)
    # Tenta converter no formato com hora
    # Tenta converter no formato com hora
    datas_convertidas = pd.to_datetime(
        df['data_do_acidente'],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )

    # Onde falhou, tenta no formato sem hora
    datas_convertidas = datas_convertidas.fillna(
        pd.to_datetime(
            df['data_do_acidente'],
            format='%Y-%m-%d',
            errors='coerce'
        )
    )

    # Imprime datas inválidas
    mask_invalidas = (
        datas_convertidas.isna()
        & df['data_do_acidente'].notna()
    )

    if mask_invalidas.any():
        print('\nDatas com erro de conversão:')

        for idx, valor in df.loc[
            mask_invalidas,
            'data_do_acidente'
        ].items():

            print(
                f'  Linha {idx + 2}: '
                f'"{valor}" '
                f'(esperado: %Y-%m-%d %H:%M:%S '
                f'ou %Y-%m-%d)'
            )

    # Salva no dataframe
    df['data_do_acidente'] = datas_convertidas

    planilhas[nome] = df
    print(f'[OK] {nome} — {len(df)} registros | instituição: {df["instituicao"].iloc[0]}')

print(f'\nTotal de arquivos carregados: {len(planilhas)}')

[OK] 2023_famar.csv — 93 registros | instituição: FAMAR
[OK] 2024_famar.csv — 83 registros | instituição: FAMAR
[OK] 2023_fumes.csv — 9 registros | instituição: FUMES
[OK] 2024_fumes.csv — 12 registros | instituição: FUMES

Total de arquivos carregados: 4


In [10]:
# ─── 4. AGREGAÇÃO DOS BIÊNIOS ─────────────────────────────────────────────────
# Biênio FAMAR (2023 + 2024)
df_famar = pd.concat(
    [planilhas[f] for f in ['2023_famar.csv', '2024_famar.csv'] if f in planilhas],
    ignore_index=True
)

# Biênio FUMES (2023 + 2024)
df_fumes = pd.concat(
    [planilhas[f] for f in ['2023_fumes.csv', '2024_fumes.csv'] if f in planilhas],
    ignore_index=True
)

print(f'Biênio FAMAR — total de registros: {len(df_famar)}')
print(f'Biênio FUMES — total de registros: {len(df_fumes)}')

Biênio FAMAR — total de registros: 176
Biênio FUMES — total de registros: 21


In [15]:
# ─── 7. ANÁLISE TRIMESTRAL DA ATIVIDADE (ENFERMAGEM vs OUTROS) ───────────────
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact


def categorizar_atividade(valor):
    """
    Enfermagem:
    - enfermagem
    - enfermeira / enfermeiro
    - técnico(a) de enfermagem
    - auxiliar de enfermagem
    e derivados textuais.

    Outros:
    - qualquer outro valor
    """
    if pd.isna(valor):
        return 'Outros'

    texto = str(valor).strip().lower()

    palavras_enfermagem = [
        'enferm',
        'enfermeir',
        'enfermeiro',
        'técnico de enfermagem',
        'tecnico de enfermagem',
        'auxiliar de enfermagem'
    ]

    if any(p in texto for p in palavras_enfermagem):
        return 'Enfermagem'

    return 'Outros'


def relatorio_fisher_atividade(df: pd.DataFrame, nome_inst: str):

    print('\n' + '=' * 90)
    print(f' {nome_inst} — BIÊNIO: ATIVIDADE (ENFERMAGEM vs OUTROS) ')
    print('=' * 90)

    df = df.copy()

    # Remove trimestre inválido
    df = df[df['trimestre'].notna()].copy()

    # Categoriza atividade
    df['atividade_categoria'] = (
        df['atividade']
        .apply(categorizar_atividade)
    )

    # Tabela absoluta
    tabela = pd.crosstab(
        df['trimestre'],
        df['atividade_categoria']
    )

    # Garante colunas
    for col in ['Enfermagem', 'Outros']:
        if col not in tabela.columns:
            tabela[col] = 0

    tabela = tabela[['Enfermagem', 'Outros']]

    # Totais por trimestre
    tabela['Total'] = tabela.sum(axis=1)

    # Percentuais
    tabela['% Enfermagem'] = (
        tabela['Enfermagem']
        / tabela['Total']
        * 100
    ).round(2)

    tabela['% Outros'] = (
        tabela['Outros']
        / tabela['Total']
        * 100
    ).round(2)

    print('\nCONTAGEM AGREGADA POR TRIMESTRE')
    print('-' * 90)

    print(
        tabela[
            [
                'Enfermagem',
                'Outros',
                'Total',
                '% Enfermagem',
                '% Outros'
            ]
        ]
    )

    # ==========================================================
    # TESTE EXATO DE FISHER ENTRE TRIMESTRES
    # ==========================================================
    print('\n' + '=' * 90)
    print('TESTE EXATO DE FISHER ENTRE TRIMESTRES')
    print('=' * 90)

    trimestres = sorted(tabela.index.tolist())

    resultados = []

    for i in range(len(trimestres)):
        for j in range(i + 1, len(trimestres)):

            t1 = trimestres[i]
            t2 = trimestres[j]

            enf_t1 = int(tabela.loc[t1, 'Enfermagem'])
            out_t1 = int(tabela.loc[t1, 'Outros'])

            enf_t2 = int(tabela.loc[t2, 'Enfermagem'])
            out_t2 = int(tabela.loc[t2, 'Outros'])

            matriz = [
                [enf_t1, out_t1],
                [enf_t2, out_t2]
            ]

            try:
                odds_ratio, p_valor = fisher_exact(matriz)
            except Exception:
                odds_ratio = np.nan
                p_valor = np.nan

            resultados.append({
                'Comparação': f'{t1} vs {t2}',

                'N Enfermagem T1': enf_t1,
                'N Outros T1': out_t1,
                '% Enfermagem T1': round(
                    enf_t1 / (enf_t1 + out_t1) * 100, 2
                ) if (enf_t1 + out_t1) > 0 else 0,

                'N Enfermagem T2': enf_t2,
                'N Outros T2': out_t2,
                '% Enfermagem T2': round(
                    enf_t2 / (enf_t2 + out_t2) * 100, 2
                ) if (enf_t2 + out_t2) > 0 else 0,

                'Odds Ratio': odds_ratio,
                'P-valor Fisher': p_valor,
                'Significativo (p<0.05)': (
                    'Sim'
                    if pd.notna(p_valor) and p_valor < 0.05
                    else 'Não'
                )
            })

    resultado_df = pd.DataFrame(resultados)

    print('\nRESULTADOS DO TESTE DE FISHER')
    print('-' * 90)

    if not resultado_df.empty:
        print(resultado_df.to_string(index=False))
    else:
        print('Sem dados suficientes para comparação.')


# ==========================================================
# EXECUÇÃO
# ==========================================================
relatorio_fisher_atividade(df_famar, 'FAMAR')
relatorio_fisher_atividade(df_fumes, 'FUMES')


 FAMAR — BIÊNIO: ATIVIDADE (ENFERMAGEM vs OUTROS) 

CONTAGEM AGREGADA POR TRIMESTRE
------------------------------------------------------------------------------------------
atividade_categoria  Enfermagem  Outros  Total  % Enfermagem  % Outros
trimestre                                                             
T1.0                         36      13     49         73.47     26.53
T2.0                         28       7     35         80.00     20.00
T3.0                         47       5     52         90.38      9.62
T4.0                         36       3     39         92.31      7.69

TESTE EXATO DE FISHER ENTRE TRIMESTRES

RESULTADOS DO TESTE DE FISHER
------------------------------------------------------------------------------------------
  Comparação  N Enfermagem T1  N Outros T1  % Enfermagem T1  N Enfermagem T2  N Outros T2  % Enfermagem T2  Odds Ratio  P-valor Fisher Significativo (p<0.05)
T1.0 vs T2.0               36           13            73.47               28  

In [12]:
# ─── 5. FUNÇÃO: TRIMESTRE A PARTIR DA DATA ────────────────────────────────────
def adicionar_trimestre(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['trimestre'] = (
        pd.to_datetime(
            df['data_do_acidente'],
            format='%Y-%m-%d %H:%M:%S',
            errors='coerce'
        )
        .dt.quarter
        .apply(lambda x: f'T{x}' if pd.notna(x) else None)
    )

    return df


df_famar = adicionar_trimestre(df_famar)
df_fumes = adicionar_trimestre(df_fumes)

print(
    'Trimestres identificados — FAMAR:',
    sorted(df_famar['trimestre'].dropna().unique())
)

print(
    'Trimestres identificados — FUMES:',
    sorted(df_fumes['trimestre'].dropna().unique())
)

Trimestres identificados — FAMAR: ['T1.0', 'T2.0', 'T3.0', 'T4.0']
Trimestres identificados — FUMES: ['T1', 'T2', 'T3', 'T4']


In [13]:
# ─── 7. RELATÓRIO TRIMESTRAL DOS BIÊNIOS ──────────────────────────────────────
def relatorio_trimestral(df: pd.DataFrame, nome_inst: str):
    total = len(df)
    print(f'\n{" " + nome_inst + " — Biênio por Trimestre ":=^60}')
    print(f'  Total do biênio: {total} registros\n')

    por_trim = (
        df.groupby('trimestre', dropna=False)
          .size()
          .reset_index(name='n')
          .sort_values('trimestre')
    )

    print(f'  {"Trimestre":<15} {"N (abs)":>10} {"% do biênio":>14}')
    print(f'  {"-"*15} {"-"*10} {"-"*14}')

    for _, row in por_trim.iterrows():
        trim = str(row['trimestre']) if pd.notna(row['trimestre']) else 'Data inválida'
        n    = int(row['n'])
        pct  = (n / total * 100) if total > 0 else 0
        print(f'  {trim:<15} {n:>10} {pct:>13.1f}%')

    print(f'  {"TOTAL":<15} {total:>10} {100.0:>13.1f}%')

relatorio_trimestral(df_famar, 'FAMAR')
relatorio_trimestral(df_fumes, 'FUMES')


=============== FAMAR — Biênio por Trimestre ===============
  Total do biênio: 176 registros

  Trimestre          N (abs)    % do biênio
  --------------- ---------- --------------
  T1.0                    49          27.8%
  T2.0                    35          19.9%
  T3.0                    52          29.5%
  T4.0                    39          22.2%
  Data inválida            1           0.6%
  TOTAL                  176         100.0%

=============== FUMES — Biênio por Trimestre ===============
  Total do biênio: 21 registros

  Trimestre          N (abs)    % do biênio
  --------------- ---------- --------------
  T1                       5          23.8%
  T2                       6          28.6%
  T3                       7          33.3%
  T4                       3          14.3%
  TOTAL                   21         100.0%


In [17]:
# ─── 7. ANÁLISE TRIMESTRAL DA ATIVIDADE (ENFERMAGEM vs OUTROS) ───────────────
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact


def categorizar_atividade(valor):
    """
    Enfermagem:
    - enfermagem
    - enfermeira / enfermeiro
    - técnico(a) de enfermagem
    - auxiliar de enfermagem
    e derivados textuais.

    Outros:
    - qualquer outro valor
    """
    if pd.isna(valor):
        return 'Outros'

    texto = str(valor).strip().lower()

    palavras_enfermagem = [
        'enferm',
        'enfermeir',
        'enfermeiro',
        'técnico de enfermagem',
        'tecnico de enfermagem',
        'auxiliar de enfermagem'
    ]

    if any(p in texto for p in palavras_enfermagem):
        return 'Enfermagem'

    return 'Outros'


def relatorio_fisher_atividade(df: pd.DataFrame, nome_inst: str):

    print('\n' + '=' * 90)
    print(f' {nome_inst} — BIÊNIO: ATIVIDADE (ENFERMAGEM vs OUTROS) ')
    print('=' * 90)

    df = df.copy()

    # Remove trimestre inválido
    df = df[df['trimestre'].notna()].copy()

    # Categoriza atividade
    df['atividade_categoria'] = (
        df['atividade']
        .apply(categorizar_atividade)
    )

    # Tabela absoluta
    tabela = pd.crosstab(
        df['trimestre'],
        df['atividade_categoria']
    )

    # Garante colunas
    for col in ['Enfermagem', 'Outros']:
        if col not in tabela.columns:
            tabela[col] = 0

    tabela = tabela[['Enfermagem', 'Outros']]

    # Totais por trimestre
    tabela['Total'] = tabela.sum(axis=1)

    # Percentuais
    tabela['% Enfermagem'] = (
        tabela['Enfermagem']
        / tabela['Total']
        * 100
    ).round(2)

    tabela['% Outros'] = (
        tabela['Outros']
        / tabela['Total']
        * 100
    ).round(2)

    print('\nCONTAGEM AGREGADA POR TRIMESTRE')
    print('-' * 90)

    print(
        tabela[
            [
                'Enfermagem',
                'Outros',
                'Total',
                '% Enfermagem',
                '% Outros'
            ]
        ]
    )

    # ==========================================================
    # TESTE EXATO DE FISHER ENTRE TRIMESTRES
    # ==========================================================
    print('\n' + '=' * 90)
    print('TESTE EXATO DE FISHER ENTRE TRIMESTRES')
    print('=' * 90)

    trimestres = sorted(tabela.index.tolist())

    resultados = []

    for i in range(len(trimestres)):
        for j in range(i + 1, len(trimestres)):

            t1 = trimestres[i]
            t2 = trimestres[j]

            enf_t1 = int(tabela.loc[t1, 'Enfermagem'])
            out_t1 = int(tabela.loc[t1, 'Outros'])

            enf_t2 = int(tabela.loc[t2, 'Enfermagem'])
            out_t2 = int(tabela.loc[t2, 'Outros'])

            matriz = [
                [enf_t1, out_t1],
                [enf_t2, out_t2]
            ]

            try:
                odds_ratio, p_valor = fisher_exact(matriz)
            except Exception:
                odds_ratio = np.nan
                p_valor = np.nan

            resultados.append({
                'Comparação': f'{t1} vs {t2}',

                'N Enfermagem T1': enf_t1,
                'N Outros T1': out_t1,
                '% Enfermagem T1': round(
                    enf_t1 / (enf_t1 + out_t1) * 100, 2
                ) if (enf_t1 + out_t1) > 0 else 0,

                'N Enfermagem T2': enf_t2,
                'N Outros T2': out_t2,
                '% Enfermagem T2': round(
                    enf_t2 / (enf_t2 + out_t2) * 100, 2
                ) if (enf_t2 + out_t2) > 0 else 0,

                'Odds Ratio': odds_ratio,
                'P-valor Fisher': p_valor,
                'Significativo (p<0.05)': (
                    'Sim'
                    if pd.notna(p_valor) and p_valor < 0.05
                    else 'Não'
                )
            })

    resultado_df = pd.DataFrame(resultados)

    print('\nRESULTADOS DO TESTE DE FISHER')
    print('-' * 90)

    if not resultado_df.empty:
        print(resultado_df.to_string(index=False))
    else:
        print('Sem dados suficientes para comparação.')


# ==========================================================
# EXECUÇÃO
# ==========================================================
relatorio_fisher_atividade(df_famar, 'FAMAR')
relatorio_fisher_atividade(df_fumes, 'FUMES')


 FAMAR — BIÊNIO: ATIVIDADE (ENFERMAGEM vs OUTROS) 

CONTAGEM AGREGADA POR TRIMESTRE
------------------------------------------------------------------------------------------
atividade_categoria  Enfermagem  Outros  Total  % Enfermagem  % Outros
trimestre                                                             
T1.0                         36      13     49         73.47     26.53
T2.0                         28       7     35         80.00     20.00
T3.0                         47       5     52         90.38      9.62
T4.0                         36       3     39         92.31      7.69

TESTE EXATO DE FISHER ENTRE TRIMESTRES

RESULTADOS DO TESTE DE FISHER
------------------------------------------------------------------------------------------
  Comparação  N Enfermagem T1  N Outros T1  % Enfermagem T1  N Enfermagem T2  N Outros T2  % Enfermagem T2  Odds Ratio  P-valor Fisher Significativo (p<0.05)
T1.0 vs T2.0               36           13            73.47               28  